# 01 · Extract raw data for the Iberian AOIs

Downloads and caches the raw inputs for the fire-spread, meteo, DEM and harmonization work, for every AOI in [`aoi.csv`](../../aoi.csv) (#22).
Part of #25. The credentials it uses were set up in #23.

| Step | Dataset | Source / access | Output (`data/raw/…`) |
|---|---|---|---|
| A | Active fires, MODIS C6.1 1 km + VIIRS 375 m (S-NPP, NOAA-20, NOAA-21) | NASA FIRMS Area API (`FIRMS_MAP_KEY`) | `firms/aoi/<aoi>_firms.csv` |
| B | Same, whole Iberia 2000–2026 (for the harmonized calendar) | FIRMS yearly country files (public) + Area API for NOAA-21 and 2026 | `firms/country/…`, `firms/api_iberia/…` |
| C | Elevation, slope, aspect, SRTM 30 m | Earth Engine `USGS/SRTMGL1_003` | `terrain/<aoi>_srtm30_terrain.tif` |
| D | Pre-fire NDVI + 2001–(year−1) climatology and anomaly, MOD13Q1 250 m | Earth Engine `MODIS/061/MOD13Q1` | `ndvi/<aoi>_mod13q1_ndvi.tif`, `ndvi/<aoi>_ndvi_timeseries.csv` |
| E | Burned area (burn day of year), MCD64A1 500 m | Earth Engine `MODIS/061/MCD64A1` | `burned_area/<aoi>_mcd64a1.tif` |
| F | Burn severity dNBR, Sentinel-2 20 m | Earth Engine `COPERNICUS/S2_SR_HARMONIZED` | `sentinel2/<aoi>_s2_dnbr.tif` |
| G | Hourly weather: ERA5-Land ~9 km and ERA5 ~28 km (gusts, 100 m and 850 hPa wind, boundary-layer height) | Earth Engine `ECMWF/ERA5_LAND/HOURLY`, `ECMWF/ERA5/HOURLY` | `weather/<aoi>_era5land_hourly.csv`, `weather/<aoi>_era5_hourly.csv` |

ERA5 replaces CERRA, so no Copernicus CDS account is needed.

**Time windows per AOI** (columns in `aoi.csv`):
- *full window* `pre_fire_od → post_fire_do`, used for FIRMS, weather and post-fire NDVI.
- *pre-fire window* `pre_fire_od → pre_fire_do` and *post-fire window* `post_fire_od → post_fire_do`, used for Sentinel-2 dNBR.
- *fire window* `pre_fire_do → post_fire_od`, used for MCD64A1.

**Before the first run**
1. `python -m venv .venv` and `.venv\Scripts\pip install -r requirements.txt` (Windows) or `.venv/bin/pip …` (Linux/macOS).
2. `.env` in the repo root with `FIRMS_MAP_KEY` and `GEE_PROJECT` (see `.env.example`).
3. Sign in to Earth Engine once: `.venv\Scripts\earthengine authenticate`. Your Google account must have access to `GEE_PROJECT`.

Re-running is safe. Files that already exist are skipped unless `OVERWRITE=1`. Everything goes to `data/raw/`, which git ignores, and `data/raw/manifest.json` records each file's dataset, window and attribution.

In [ ]:
import datetime as dt
import io
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "aoi.csv").exists())
load_dotenv(REPO / ".env")
RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
IBERIA_BBOX = (-9.6, 36.0, 3.4, 43.8)  # W, S, E, N  (#22)

# ---- Run configuration. Environment variables override the defaults, which is handy for headless runs. ----
AOI_IDS = os.getenv("AOI_IDS", "all")  # "all" or comma-separated aoi_id values
ALL_STEPS = ["firms_aoi", "firms_iberia_archive", "terrain", "ndvi", "burned_area", "sentinel2_dnbr", "weather"]
STEPS = set(os.getenv("STEPS", ",".join(ALL_STEPS)).split(","))
ARCHIVE_YEARS = range(int(os.getenv("ARCHIVE_FROM", 2000)), int(os.getenv("ARCHIVE_TO", 2026)) + 1)
OVERWRITE = os.getenv("OVERWRITE", "0") == "1"

for var in ("FIRMS_MAP_KEY", "GEE_PROJECT"):
    assert os.getenv(var), f"{var} missing in {REPO / '.env'}"
print(f"repo: {REPO}\nsteps: {sorted(STEPS)}\nAOIs: {AOI_IDS} | archive years: {ARCHIVE_YEARS.start}-{ARCHIVE_YEARS.stop - 1} | overwrite: {OVERWRITE}")

### Helpers: caching and manifest

In [ ]:
MANIFEST_PATH = RAW / "manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

ATTRIBUTION = {
    "firms": "We acknowledge the use of data and imagery from NASA's Fire Information for Resource Management System (FIRMS) (https://www.earthdata.nasa.gov/firms), part of NASA's Earth Science Data and Information System (ESDIS).",
    "srtm": "NASA/USGS Shuttle Radar Topography Mission (SRTM) 1 arc-second Global (USGS/SRTMGL1_003), via Google Earth Engine.",
    "mod13q1": "Didan, K. MODIS/Terra Vegetation Indices 16-Day L3 Global 250m SIN Grid V061. NASA EOSDIS Land Processes DAAC.",
    "mcd64a1": "Giglio, L., Justice, C., Boschetti, L., Roy, D. MODIS/Terra+Aqua Burned Area Monthly L3 Global 500m SIN Grid V061. NASA EOSDIS Land Processes DAAC.",
    "sentinel2": "Contains modified Copernicus Sentinel data, processed in Google Earth Engine.",
    "era5": "Contains modified Copernicus Climate Change Service information (ERA5 and ERA5-Land hourly reanalysis), via Google Earth Engine.",
}


def record(path, dataset, attribution, aoi_id=None, **meta):
    # Add or refresh a file's entry in the manifest.
    manifest[path.relative_to(REPO).as_posix()] = {
        "dataset": dataset, "aoi_id": aoi_id, "bytes": path.stat().st_size,
        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "attribution": ATTRIBUTION[attribution], **meta,
    }


def save_manifest():
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def cached(path):
    # True if the file already exists (so the download is skipped); otherwise create its folder.
    if path.exists() and path.stat().st_size > 0 and not OVERWRITE:
        print(f"  cached   {path.relative_to(REPO)}")
        return True
    path.parent.mkdir(parents=True, exist_ok=True)
    return False


def saved(path):
    print(f"  saved    {path.relative_to(REPO)}  ({path.stat().st_size / 1e6:.1f} MB)")


def day(ts):
    return pd.Timestamp(ts).strftime("%Y-%m-%d")

## AOIs from `aoi.csv`

In [ ]:
DATE_COLS = ["pre_fire_od", "pre_fire_do", "post_fire_od", "post_fire_do"]
aois = pd.read_csv(REPO / "aoi.csv", encoding="utf-8-sig", parse_dates=DATE_COLS)
aois["bbox"] = list(zip(aois.bbox_w, aois.bbox_s, aois.bbox_e, aois.bbox_n))
aois["utm_epsg"] = 32600 + aois.strefa_utm.str[:2].astype(int)  # first listed zone, northern hemisphere

# Sanity checks: ignition inside its bbox, bbox inside Iberia, windows in order
w, s, e, n = IBERIA_BBOX
problems = aois[
    ~aois.zaplon_lon.between(aois.bbox_w, aois.bbox_e) | ~aois.zaplon_lat.between(aois.bbox_s, aois.bbox_n)
    | (aois.bbox_w < w) | (aois.bbox_e > e) | (aois.bbox_s < s) | (aois.bbox_n > n)
    | ~((aois.pre_fire_od < aois.pre_fire_do) & (aois.pre_fire_do < aois.post_fire_od) & (aois.post_fire_od < aois.post_fire_do))
]
assert problems.empty, f"invalid AOIs: {problems.aoi_id.tolist()}"

if AOI_IDS != "all":
    wanted = AOI_IDS.split(",")
    unknown = set(wanted) - set(aois.aoi_id)
    assert not unknown, f"unknown aoi_id: {unknown}"
    aois = aois[aois.aoi_id.isin(wanted)].reset_index(drop=True)

aois[["aoi_id", "pozar", "rok", "bbox_w", "bbox_s", "bbox_e", "bbox_n", *DATE_COLS, "strefa_utm", "utm_epsg"]]

## A · FIRMS active fires per AOI

Each sensor is requested separately. The science-quality archive (`*_SP`) is used wherever it covers the day, and near-real-time (`*_NRT`) only after the archive ends, as reported live by FIRMS `data_availability`.
The Area API returns at most 5 days per request, and the MAP_KEY allows 5000 requests per 10 minutes, shared by the whole team.

In [ ]:
FIRMS = "https://firms.modaps.eosdis.nasa.gov"
MAP_KEY = os.environ["FIRMS_MAP_KEY"]
SENSORS = ["MODIS", "VIIRS_SNPP", "VIIRS_NOAA20", "VIIRS_NOAA21"]

availability = pd.read_csv(f"{FIRMS}/api/data_availability/csv/{MAP_KEY}/ALL", parse_dates=["min_date", "max_date"]).set_index("data_id")


def firms_source(sensor, d):
    # Return the FIRMS source id covering day d, preferring SP over NRT (None if neither does).
    for kind in ("SP", "NRT"):
        sid = f"{sensor}_{kind}"
        if sid in availability.index and availability.at[sid, "min_date"] <= d <= availability.at[sid, "max_date"]:
            return sid
    return None


def firms_get(source, bbox, start, n_days):
    url = f"{FIRMS}/api/area/csv/{MAP_KEY}/{source}/{','.join(map(str, bbox))}/{n_days}/{day(start)}"
    for attempt in range(4):
        r = requests.get(url, timeout=300)
        if r.status_code == 200 and r.text.startswith("latitude"):
            return pd.read_csv(io.StringIO(r.text), dtype={"confidence": str, "acq_time": str})
        time.sleep(5 * (attempt + 1))  # rate limit or transient error: back off
    # Never echo the URL, because it contains the MAP_KEY
    raise RuntimeError(f"FIRMS {source} {day(start)}+{n_days}d failed: HTTP {r.status_code} {r.text[:150]!r}")


def firms_area(bbox, start, end, sensors=SENSORS):
    # All detections in bbox for [start, end] inclusive, one request per <=5 days per sensor.
    days = pd.date_range(start, end, freq="D")
    frames = []
    for sensor in sensors:
        src = pd.Series([firms_source(sensor, d) or "-" for d in days], index=days)
        for source, run in src.groupby((src != src.shift()).cumsum()):
            source = run.iloc[0]
            if source == "-":
                continue
            for i in range(0, len(run), 5):
                chunk = run.iloc[i:i + 5]
                df = firms_get(source, bbox, chunk.index[0], len(chunk))
                if len(df):
                    frames.append(df.assign(source=source, sensor=sensor))
                time.sleep(0.3)
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["acq_datetime_utc"] = pd.to_datetime(df.acq_date + " " + df.acq_time.str.zfill(4), format="%Y-%m-%d %H%M", utc=True)
    return df.sort_values("acq_datetime_utc").reset_index(drop=True)


availability

In [ ]:
if "firms_aoi" in STEPS:
    for aoi in aois.itertuples():
        path = RAW / "firms" / "aoi" / f"{aoi.aoi_id}_firms.csv"
        print(aoi.aoi_id)
        if cached(path):
            continue
        df = firms_area(aoi.bbox, aoi.pre_fire_od, aoi.post_fire_do)
        df.to_csv(path, index=False)
        saved(path)
        record(path, "NASA FIRMS active fires (MODIS C6.1 1 km, VIIRS 375 m)", "firms", aoi.aoi_id,
               window=[day(aoi.pre_fire_od), day(aoi.post_fire_do)], bbox=aoi.bbox,
               rows_per_source=df.source.value_counts().to_dict() if len(df) else {})
        print("   ", df.source.value_counts().to_dict() if len(df) else "no detections")
    save_manifest()

## B · FIRMS archive for the whole Iberian Peninsula (harmonized calendar)

- **2000–2025, MODIS / VIIRS S-NPP / VIIRS NOAA-20:** FIRMS publishes yearly CSVs per country. They're public and need no key, and we download them for Spain and Portugal.
- **VIIRS NOAA-21 (from 2023) and all of 2026:** not in the yearly files yet, so they come from the Area API over the Iberia bbox.

The two sources have different footprints. The country files cover Spain and Portugal, including islands outside the bbox. The Area API covers the bbox, including a strip of southern France and Andorra. Clip both to the same mainland polygon before harmonizing.

In [ ]:
COUNTRY_PRODUCTS = {"modis": ("MODIS", 2000), "viirs-snpp": ("VIIRS_SNPP", 2012), "viirs-jpss1": ("VIIRS_NOAA20", 2018)}
COUNTRIES = ["Spain", "Portugal"]


def download(url, path, timeout=600):
    with requests.get(url, stream=True, timeout=timeout) as r:
        if r.status_code == 404:
            return False
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return True


if "firms_iberia_archive" in STEPS:
    # 1) yearly country files
    for product, (sensor, first_year) in COUNTRY_PRODUCTS.items():
        for year in ARCHIVE_YEARS:
            if year < first_year:
                continue
            for country in COUNTRIES:
                path = RAW / "firms" / "country" / product / f"{product}_{year}_{country}.csv"
                if cached(path):
                    continue
                if download(f"{FIRMS}/data/country/{product}/{year}/{product}_{year}_{country}.csv", path):
                    saved(path)
                    record(path, f"NASA FIRMS yearly country file ({sensor})", "firms", year=year, country=country)
                else:
                    print(f"  missing  {product} {year} {country} (not published yet; covered by the Area API below)")

    # 2) gaps through the Area API: NOAA-21 for every year, every sensor for years without country files
    today = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
    for year in ARCHIVE_YEARS:
        country_file_exists = (RAW / "firms" / "country" / "modis" / f"modis_{year}_Spain.csv").exists()
        sensors = ["VIIRS_NOAA21"] if country_file_exists else SENSORS
        for sensor in sensors:
            start, end = pd.Timestamp(f"{year}-01-01"), min(pd.Timestamp(f"{year}-12-31"), today)
            if not any(firms_source(sensor, d) for d in pd.date_range(start, end, freq="MS").union([end])):
                continue
            path = RAW / "firms" / "api_iberia" / f"{sensor}_{year}.csv"
            if cached(path):
                continue
            df = firms_area(IBERIA_BBOX, start, end, sensors=[sensor])
            df.to_csv(path, index=False)
            saved(path)
            record(path, f"NASA FIRMS Area API, Iberia bbox ({sensor})", "firms", year=year, bbox=IBERIA_BBOX,
                   window=[day(start), day(end)], rows_per_source=df.source.value_counts().to_dict() if len(df) else {})
    save_manifest()

## Google Earth Engine setup

Rasters are downloaded directly as GeoTIFF in the AOI's UTM zone (`utm_epsg`; AOIs spanning 29T and 30T use the first zone). Nothing is exported to Cloud Storage, so there are no storage costs. A single download is limited to about 48 MB, which the AOIs in `aoi.csv` stay under.

In [ ]:
import ee

ee.Initialize(project=os.environ["GEE_PROJECT"])


def ee_region(aoi):
    return ee.Geometry.Rectangle(list(aoi.bbox), proj="EPSG:4326", geodesic=False)


def ee_download(image, aoi, scale, path):
    url = image.getDownloadURL({"region": ee_region(aoi), "scale": scale, "crs": f"EPSG:{aoi.utm_epsg}", "format": "GEO_TIFF"})
    if not download(url, path):
        raise RuntimeError(f"Earth Engine download failed for {path.name}")
    saved(path)


print("Earth Engine OK, project:", os.environ["GEE_PROJECT"])

## C · Terrain: elevation (m), slope (°), aspect (° from north), SRTM 30 m

In [ ]:
if "terrain" in STEPS:
    terrain = ee.Terrain.products(ee.Image("USGS/SRTMGL1_003")).select(["elevation", "slope", "aspect"]).toFloat()
    for aoi in aois.itertuples():
        path = RAW / "terrain" / f"{aoi.aoi_id}_srtm30_terrain.tif"
        print(aoi.aoi_id)
        if cached(path):
            continue
        ee_download(terrain, aoi, 30, path)
        record(path, "SRTM 30 m elevation/slope/aspect (bands: elevation, slope, aspect)", "srtm", aoi.aoi_id,
               scale_m=30, crs=f"EPSG:{aoi.utm_epsg}")
    save_manifest()

## D · NDVI: pre-fire state, climatology and anomaly, MOD13Q1 250 m

The GeoTIFF bands are:
- `ndvi_pre`: median of the 16-day composites in the 48 days before `pre_fire_do`.
- `ndvi_clim`, `ndvi_clim_std`: the same calendar window across 2001 to (fire year − 1).
- `ndvi_anom`, `ndvi_z`: `ndvi_pre − ndvi_clim`, and the same divided by `ndvi_clim_std`.
- `ndvi_post`: median of the composites in the post-fire window.

Only pixels with `SummaryQA ≤ 1` (good or marginal) are used, with scale factor 0.0001.
The CSV is the AOI-mean NDVI for every composite from 2000 to today, which feeds the inertia and anomaly work.

In [ ]:
MOD13 = ee.ImageCollection("MODIS/061/MOD13Q1")


def ndvi_clean(img):
    return ee.Image(img.select("NDVI").multiply(0.0001).updateMask(img.select("SummaryQA").lte(1))
                    .rename("ndvi").copyProperties(img, ["system:time_start"]))


if "ndvi" in STEPS:
    for aoi in aois.itertuples():
        print(aoi.aoi_id)
        region = ee_region(aoi)

        tif = RAW / "ndvi" / f"{aoi.aoi_id}_mod13q1_ndvi.tif"
        if not cached(tif):
            pre_end = aoi.pre_fire_do + pd.Timedelta(days=1)
            pre_start = aoi.pre_fire_do - pd.Timedelta(days=48)
            pre = MOD13.filterDate(day(pre_start), day(pre_end)).map(ndvi_clean).median()
            doy0, doy1 = pre_start.dayofyear, aoi.pre_fire_do.dayofyear
            clim_col = (MOD13.filter(ee.Filter.calendarRange(2001, aoi.rok - 1, "year"))
                        .filter(ee.Filter.calendarRange(doy0, doy1, "day_of_year")).map(ndvi_clean))
            clim, clim_std = clim_col.median(), clim_col.reduce(ee.Reducer.stdDev())
            post = MOD13.filterDate(day(aoi.post_fire_od), day(aoi.post_fire_do + pd.Timedelta(days=1))).map(ndvi_clean).median()
            img = ee.Image.cat([
                pre.rename("ndvi_pre"), clim.rename("ndvi_clim"), clim_std.rename("ndvi_clim_std"),
                pre.subtract(clim).rename("ndvi_anom"), pre.subtract(clim).divide(clim_std).rename("ndvi_z"),
                post.rename("ndvi_post"),
            ]).toFloat()
            ee_download(img, aoi, 250, tif)
            record(tif, "MOD13Q1 NDVI pre/clim/anomaly/post (bands: ndvi_pre, ndvi_clim, ndvi_clim_std, ndvi_anom, ndvi_z, ndvi_post)",
                   "mod13q1", aoi.aoi_id, scale_m=250, crs=f"EPSG:{aoi.utm_epsg}",
                   pre_window=[day(pre_start), day(aoi.pre_fire_do)], clim_years=[2001, int(aoi.rok) - 1],
                   post_window=[day(aoi.post_fire_od), day(aoi.post_fire_do)])

        csv = RAW / "ndvi" / f"{aoi.aoi_id}_ndvi_timeseries.csv"
        if not cached(csv):
            reducer = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)
            fc = MOD13.filterDate("2000-01-01", day(pd.Timestamp.now())).map(
                lambda img: ee.Feature(None, ndvi_clean(img).reduceRegion(reducer, region, 250, maxPixels=int(1e9)))
                .set("date", img.date().format("YYYY-MM-dd")))
            ts = pd.DataFrame([f["properties"] for f in fc.getInfo()["features"]]).rename(
                columns={"ndvi_mean": "ndvi_mean", "ndvi_count": "valid_pixels"})
            ts.sort_values("date").to_csv(csv, index=False)
            saved(csv)
            record(csv, "MOD13Q1 AOI-mean NDVI per 16-day composite (QA <= 1)", "mod13q1", aoi.aoi_id, rows=len(ts))
    save_manifest()

## E · Burned area: MCD64A1 burn day of year, 500 m

Bands: `burn_doy` is the latest burn day in the fire-window months (unburned pixels are masked), and `uncertainty_days` is the product's uncertainty.

In [ ]:
MCD64 = ee.ImageCollection("MODIS/061/MCD64A1")

if "burned_area" in STEPS:
    for aoi in aois.itertuples():
        path = RAW / "burned_area" / f"{aoi.aoi_id}_mcd64a1.tif"
        print(aoi.aoi_id)
        if cached(path):
            continue
        start = aoi.pre_fire_do.replace(day=1)
        col = MCD64.filterDate(day(start), day(aoi.post_fire_od + pd.Timedelta(days=1)))
        if col.size().getInfo() == 0:
            print("  skipped  no MCD64A1 months on Earth Engine for this window yet")
            continue
        burn = col.select("BurnDate").max()
        img = ee.Image.cat([burn.rename("burn_doy"), col.select("Uncertainty").max().rename("uncertainty_days")]) \
            .updateMask(burn.gt(0)).toInt16()
        ee_download(img, aoi, 500, path)
        record(path, "MCD64A1 burned area (bands: burn_doy, uncertainty_days)", "mcd64a1", aoi.aoi_id,
               scale_m=500, crs=f"EPSG:{aoi.utm_epsg}", months=[day(start), day(aoi.post_fire_od)])
    save_manifest()

## F · Burn severity: Sentinel-2 dNBR, 20 m

NBR = (B8A − B12) / (B8A + B12), from the cloud-masked median of each window. Pixels with SCL classes for no data, saturated, cloud shadow, cloud or cirrus are dropped.
The bands `nbr_pre`, `nbr_post` and `dnbr = nbr_pre − nbr_post` are stored as **int16 × 1000**.
Common dNBR severity thresholds (Key & Benson) are below 100 unburned, 100–269 low, 270–439 moderate-low, 440–659 moderate-high and 660 or more high.
Read the `uwagi` column in `aoi.csv`: for `losacio_2022_lipiec` the pre-fire window already includes the June 2022 scar.

In [ ]:
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
BAD_SCL = [0, 1, 3, 8, 9, 10]


def s2_nbr(region, start, end):
    col = (S2.filterBounds(region).filterDate(day(start), day(end + pd.Timedelta(days=1)))
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70)))
    nbr = col.map(lambda img: img.updateMask(img.select("SCL").remap(BAD_SCL, [0] * len(BAD_SCL), 1))
                  .normalizedDifference(["B8A", "B12"]).rename("nbr")).median()
    return nbr, col.size()


if "sentinel2_dnbr" in STEPS:
    for aoi in aois.itertuples():
        path = RAW / "sentinel2" / f"{aoi.aoi_id}_s2_dnbr.tif"
        print(aoi.aoi_id)
        if cached(path):
            continue
        region = ee_region(aoi)
        pre, n_pre = s2_nbr(region, aoi.pre_fire_od, aoi.pre_fire_do)
        post, n_post = s2_nbr(region, aoi.post_fire_od, aoi.post_fire_do)
        n_pre, n_post = n_pre.getInfo(), n_post.getInfo()
        if not (n_pre and n_post):
            print(f"  skipped  scenes pre={n_pre} post={n_post}")
            continue
        img = ee.Image.cat([pre.rename("nbr_pre"), post.rename("nbr_post"), pre.subtract(post).rename("dnbr")]) \
            .multiply(1000).round().toInt16()
        ee_download(img, aoi, 20, path)
        record(path, "Sentinel-2 L2A NBR/dNBR x1000 (bands: nbr_pre, nbr_post, dnbr)", "sentinel2", aoi.aoi_id,
               scale_m=20, crs=f"EPSG:{aoi.utm_epsg}", scenes_pre=n_pre, scenes_post=n_post,
               pre_window=[day(aoi.pre_fire_od), day(aoi.pre_fire_do)], post_window=[day(aoi.post_fire_od), day(aoi.post_fire_do)])
        print(f"           scenes pre={n_pre} post={n_post}")
    save_manifest()

## G · Hourly weather: ERA5-Land and ERA5, which replace CERRA

One row per grid cell per hour, covering every reanalysis cell whose centre lies in the AOI buffered by half a cell, over the full window.
- **ERA5-Land** (~9 km): 10 m wind, 2 m temperature and dew point, hourly precipitation, top-layer soil moisture.
- **ERA5** (~28 km): adds **instantaneous 10 m gusts**, **100 m wind** (closer to the height that drives crown fires), **850 hPa wind** (above the terrain) and **boundary-layer height**.

Derived columns: `*_speed_ms`, `*_dir_from_deg` (meteorological convention, the direction the wind blows *from*), `t2m_c`, `rh_pct` and precipitation in mm.

In [ ]:
WEATHER = {
    "era5land": ("ECMWF/ERA5_LAND/HOURLY", 11132, [
        "u_component_of_wind_10m", "v_component_of_wind_10m", "temperature_2m", "dewpoint_temperature_2m",
        "total_precipitation_hourly", "volumetric_soil_water_layer_1"]),
    "era5": ("ECMWF/ERA5/HOURLY", 27830, [
        "u_component_of_wind_10m", "v_component_of_wind_10m", "u_component_of_wind_100m", "v_component_of_wind_100m",
        "instantaneous_10m_wind_gust", "u_component_of_wind_850hPa", "v_component_of_wind_850hPa",
        "boundary_layer_height", "temperature_2m", "dewpoint_temperature_2m", "total_precipitation"]),
}


def hourly_table(collection_id, scale, bands, region, start, end):
    frames = []
    for s in pd.date_range(start, end, freq="7D"):
        e = min(s + pd.Timedelta(days=7), end)
        rows = ee.ImageCollection(collection_id).filterDate(day(s), day(e)).select(bands).getRegion(region, scale).getInfo()
        frames.append(pd.DataFrame(rows[1:], columns=rows[0]))
    df = pd.concat(frames, ignore_index=True).drop(columns="id")
    df.insert(0, "time_utc", pd.to_datetime(df.pop("time"), unit="ms", utc=True))
    return df.sort_values(["time_utc", "latitude", "longitude"]).reset_index(drop=True)


def add_derived(df):
    for level in ("10m", "100m", "850hPa"):
        u, v = f"u_component_of_wind_{level}", f"v_component_of_wind_{level}"
        if u in df:
            df[f"wind_{level}_speed_ms"] = np.hypot(df[u], df[v])
            df[f"wind_{level}_dir_from_deg"] = (np.degrees(np.arctan2(-df[u], -df[v])) + 360) % 360
    t, td = df.temperature_2m - 273.15, df.dewpoint_temperature_2m - 273.15
    magnus = lambda x: np.exp(17.62 * x / (243.12 + x))
    df["t2m_c"], df["rh_pct"] = t, 100 * magnus(td) / magnus(t)
    for col in ("total_precipitation_hourly", "total_precipitation"):
        if col in df:
            df["precip_mm"] = df[col] * 1000
    return df


if "weather" in STEPS:
    for aoi in aois.itertuples():
        print(aoi.aoi_id)
        for name, (cid, scale, bands) in WEATHER.items():
            path = RAW / "weather" / f"{aoi.aoi_id}_{name}_hourly.csv"
            if cached(path):
                continue
            region = ee_region(aoi).buffer(scale / 2, maxError=100).bounds(maxError=100)
            end = aoi.post_fire_do + pd.Timedelta(days=1)
            df = add_derived(hourly_table(cid, scale, bands, region, aoi.pre_fire_od, end))
            df.to_csv(path, index=False)
            saved(path)
            record(path, f"{cid} hourly at native grid cells", "era5", aoi.aoi_id, grid_m=scale,
                   cells=int(df.groupby(["latitude", "longitude"]).ngroups), hours=int(df.time_utc.nunique()),
                   window=[day(aoi.pre_fire_od), day(aoi.post_fire_do)])
            print(f"           {name}: {df.groupby(['latitude', 'longitude']).ngroups} cells x {df.time_utc.nunique()} h")
    save_manifest()

## Summary

In [ ]:
save_manifest()
summary = pd.DataFrame.from_dict(manifest, orient="index")
summary["MB"] = (summary.bytes / 1e6).round(2)
summary["aoi_id"] = summary.aoi_id.fillna("(Iberia)")
print(f"{len(summary)} files, {summary.MB.sum():.0f} MB in {RAW.relative_to(REPO)}")
summary.groupby(["aoi_id", "dataset"]).MB.sum().unstack(0).fillna(0).round(1)